<a href="https://colab.research.google.com/github/biolographer/NanoparticlesSAM/blob/main/model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Base vs. fine-tuned model comparison

Runs the base SAM2 checkpoint and the fine-tuned checkpoint over the held-out
`data/validation data/` set using the real automatic-detection inference path
(`SAM2AutomaticMaskGenerator` + `sphere_segmentation`, same as `sam2_predictor.ipynb`),
scores each against the manually-drawn ground-truth circles via
`model_comparison.py`, and compares the two models with a paired Wilcoxon
signed-rank test (per image, so within-image particle correlation doesn't
inflate significance).

This notebook is a thin GPU-dependent wrapper - all of the matching/metric/
statistics logic lives in `NanoparticlesSAM/model_comparison.py` and is unit
tested in `tests/test_model_comparison.py` without needing SAM2 or a GPU.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
nb_path = '/content/notebooks'
models_path = '/content/checkpoints/'
sys.path.insert(0, nb_path)

In [ ]:
using_colab = True

if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    get_ipython().system("{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam2.git'")

    get_ipython().system('mkdir -p $models_path')
    # base (pretrained, not fine-tuned) checkpoint
    get_ipython().system('wget -nc -P $models_path https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt')
    # fine-tuned checkpoint, copied from Drive (adjust path to wherever it was saved by sam2_training.ipynb)
    get_ipython().system('cp /content/drive/MyDrive/Colab\\ Notebooks/models/sam2.1_hiera_tiny_finetune_model_600.torch.pt $models_path')

In [ ]:
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

import os
import sys

if not os.path.isdir('./NanoparticlesSAM'):
    get_ipython().system('git clone https://github.com/Biolographer/NanoparticlesSAM.git')

module_path = os.path.abspath(os.path.join('NanoparticlesSAM/NanoparticlesSAM'))
sys.path.append(module_path)

In [ ]:
from particle_seg import sphere_segmentation
from model_comparison import evaluate_model_on_validation_set, compare_models_wilcoxon

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## Build the base and fine-tuned mask generators

Two *separate* `sam2` model instances are built here on purpose: loading the
fine-tuned `state_dict` mutates the model object in place, so reusing one
instance for both would leave the "base" model fine-tuned too.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

base_checkpoint = "/content/checkpoints/sam2.1_hiera_tiny.pt"
finetuned_checkpoint = "/content/checkpoints/sam2.1_hiera_tiny_finetune_model_600.torch.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"

# base model: pretrained weights only
sam2_base = build_sam2(model_cfg, base_checkpoint, device=device, apply_postprocessing=False)
mask_generator_base = SAM2AutomaticMaskGenerator(sam2_base)

# fine-tuned model: a second, independent instance with the trained weights loaded
sam2_finetuned = build_sam2(model_cfg, base_checkpoint, device=device, apply_postprocessing=False)
predictor_finetuned = SAM2ImagePredictor(sam2_finetuned)
state_dict = torch.load(finetuned_checkpoint, map_location="cpu")
predictor_finetuned.model.load_state_dict(state_dict)
mask_generator_finetuned = SAM2AutomaticMaskGenerator(sam2_finetuned)

## Run both models on the held-out validation set

Uses the same `sphere_segmentation` settings as production inference
(`sam2_predictor.ipynb`) so the comparison reflects real deployment behavior.
Adjust `sphere_segmentation_kwargs` to match whatever settings you actually run
in production.

In [ ]:
validation_dir = os.path.join(os.getcwd(), 'data', 'validation data')

# Same defaults used in sam2_predictor.ipynb's production run - update to match
# whatever settings you actually deploy with.
sphere_segmentation_kwargs = dict(
    min_diameter_cutoff=150,
    max_diameter_cutoff=300,
    circularity_cutoff=0.65,
    border_cutoff=True,
    max_feret_filter=False,
    min_feret_filter=False,
    hough_circles=False,
)

# Validation tifs carry no JEOL pixel-calibration metadata, so this stays in
# pixel units (nanometer_per_pixel=None) - fine for a same-image, same-units
# base-vs-finetuned comparison. Set it if you know the calibration for these images.
nanometer_per_pixel = None

base_df = evaluate_model_on_validation_set(
    validation_dir, mask_generator_base,
    nanometer_per_pixel=nanometer_per_pixel,
    sphere_segmentation_kwargs=sphere_segmentation_kwargs,
)
finetuned_df = evaluate_model_on_validation_set(
    validation_dir, mask_generator_finetuned,
    nanometer_per_pixel=nanometer_per_pixel,
    sphere_segmentation_kwargs=sphere_segmentation_kwargs,
)

print(f"{len(base_df)} validation images evaluated")
base_df

## Compare with a paired Wilcoxon signed-rank test

Paired at the image level (not particle level) to avoid within-image
correlation inflating significance. This is a standalone `scipy.stats.wilcoxon`
call in `model_comparison.py` - unrelated to `particle_stats.py`'s Welch's
t-test, which stays untouched for its own experiment-condition comparisons.

`mean_abs_radius_error` is the headline "utility" metric (did fine-tuning
make particle sizing more accurate?); precision/recall/f1 show whether
detection coverage changed too.

In [ ]:
metrics_to_compare = ['mean_abs_radius_error', 'precision', 'recall', 'f1']

comparison_results = pd.DataFrame([
    compare_models_wilcoxon(base_df, finetuned_df, metric=metric)
    for metric in metrics_to_compare
])
comparison_results

## Save results

In [ ]:
results_dir = os.path.join(os.getcwd(), 'results', 'model_comparison')
os.makedirs(results_dir, exist_ok=True)

base_df.to_pickle(os.path.join(results_dir, 'base_model_per_image.pkl'))
finetuned_df.to_pickle(os.path.join(results_dir, 'finetuned_model_per_image.pkl'))
comparison_results.to_pickle(os.path.join(results_dir, 'wilcoxon_comparison.pkl'))

print(f"saved results to {results_dir}")

## Visualize the per-image comparison

Paired lines (one per validation image) make the "showcase" case visually:
if fine-tuning helped, most lines slope the same direction.

In [ ]:
merged = base_df.merge(finetuned_df, on='img_name', suffixes=('_base', '_finetuned'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for _, row in merged.iterrows():
    axes[0].plot(['base', 'fine-tuned'],
                 [row['mean_abs_radius_error_base'], row['mean_abs_radius_error_finetuned']],
                 marker='o', color='gray', alpha=0.6)
axes[0].set_ylabel('mean abs. radius error (px)')
axes[0].set_title('Per-image radius error')

for _, row in merged.iterrows():
    axes[1].plot(['base', 'fine-tuned'], [row['f1_base'], row['f1_finetuned']],
                 marker='o', color='gray', alpha=0.6)
axes[1].set_ylabel('F1 (detection vs. annotated)')
axes[1].set_title('Per-image detection F1')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'base_vs_finetuned_per_image.png'), dpi=150)
plt.show()